In [1]:
import os
import pandas as pd
from ratelimit import limits, RateLimitException
from backoff import on_exception, expo
import requests

In [ ]:
EPOCH_TIME_JULY2026 = 1782867600
RANKED_SOLO = 420
ONE_SECOND = 1
TWO_MINUTES = 120
NUM_CHAMPIONS_PER_GAME = 10

master_division_url = 'https://oc1.api.riotgames.com/lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5' # should we add support for other regions?
matches_by_player_url = 'https://sea.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?startTime={start_time}&queue={queue}&type=ranked&start=0&count=5'
match_data_from_matchid = 'https://sea.api.riotgames.com/lol/match/v5/matches/{matchId}'
api_key = os.getenv("RIOT_API_KEY")
headers = {
    'X-Riot-Token': api_key
}


In [4]:
@on_exception(expo, RateLimitException, max_tries=8)
@limits(calls=100, period=TWO_MINUTES)
@limits(calls=20, period=ONE_SECOND)
def call_api(url, headers):
    response = requests.get(url, headers=headers)
    return response

In [5]:
player_data = call_api(master_division_url, headers)

In [12]:
num_players = len(player_data.json()['entries'])
player_id = [player_data.json()['entries'][i]['puuid'] for i in range(num_players)] # collects all the player puuids from the master division

match_data = []
for puuid in player_id:
    match_data.append(call_api(matches_by_player_url.format(puuid=puuid, start_time=EPOCH_TIME_JULY2026, queue=RANKED_SOLO), headers).json()) # collects match ids from players
    if len(match_data) > 10:
        break


In [19]:
flat_match_ids = [match for sublist in match_data for match in sublist] # flattens the list of lists into a single list of match ids
flat_match_ids = list(set(flat_match_ids)) # removes duplicates from the list of match ids

In [ ]:
champ_data = []
for match_id in flat_match_ids:
    game_data = call_api(match_data_from_matchid.format(matchId=match_id), headers)
    current_champs = []
    for i in range(NUM_CHAMPIONS_PER_GAME):
        current_champs.append(game_data.json()['info']['participants'][i]['championName']) # first 5 participants are team 1, next 5 are team 2
    champ_data.append(current_champs)

In [2]:
print(game_data.json()['info']['participants'][5]['teamPosition'])


NameError: name 'game_data' is not defined